In [ ]:
import os, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import feature_extraction as fe 
np.random.seed(42)

In [ ]:
IMAGES_DIR = "ISIC2018_Task3_Training_Input"          # pasta com os .jpg
MASKS_DIR  = "ISIC2018_Task1_Training_GroundTruth"     # opcional (ou None)
LABELS_CSV = "ISIC2018_Task3_Training_GroundTruth.csv" # one-hot das 7 classes
FEATURES_NPZ = "isic_features.npz"

USE_SHAPE = os.path.isdir(MASKS_DIR) if MASKS_DIR else False
print("Usar forma (máscaras encontradas):", USE_SHAPE)

In [ ]:
if not os.path.exists(FEATURES_NPZ):
    cfg = fe.FeatureConfig(use_shape=USE_SHAPE)
    fe.build_feature_database(
        images_dir=IMAGES_DIR,
        masks_dir=MASKS_DIR if USE_SHAPE else None,
        cfg=cfg,
        out_path=FEATURES_NPZ,
    )
else:
    print(f"{FEATURES_NPZ} já existe — pulando extração.")

Carregar


In [ ]:
data = np.load(FEATURES_NPZ, allow_pickle=True)
ids = data["ids"]
blocks = {k: data[k] for k in data.files if k != "ids"}
print("Blocos:", {k: v.shape for k, v in blocks.items()})
print("N imagens:", len(ids))

In [ ]:
# Rótulos: CSV one-hot do Task 3 -> nome da classe por id
gt = pd.read_csv(LABELS_CSV)
class_cols = [c for c in gt.columns if c != "image"]
gt["label"] = gt[class_cols].values.argmax(axis=1)
id2label = dict(zip(gt["image"], gt["label"]))

labels = np.array([id2label.get(i, -1) for i in ids])
mask_valid = labels >= 0
print("Com rótulo:", mask_valid.sum(), "/", len(labels))
print("Distribuição:", dict(zip(*np.unique(labels[mask_valid], return_counts=True))))

Normalização

In [ ]:
from sklearn.preprocessing import StandardScaler

def normalize_block(name, X):
    if name in ("color", "lbp"):      # histogramas: mantém distribuição
        return X
    return StandardScaler().fit_transform(X)

norm_blocks = {k: normalize_block(k, v) for k, v in blocks.items()}

Matrizes

In [ ]:
mport numpy as np
from sklearn.metrics import pairwise_distances

# 1. Definição das métricas por bloco de características
METRIC = {
    "deep": "cosine", 
    "color": "chi2", 
    "lbp": "chi2",
    "gabor": "euclidean", 
    "glcm": "euclidean", 
    "shape": "euclidean"
}

def chi2_distance(X):
    # Garante a entrada em float32 para economizar 50% de espaço em relação ao float64
    X = np.asarray(X, dtype=np.float32)
    n = X.shape[0]
    D = np.zeros((n, n), dtype=np.float32)
    
    # Loop com reaproveitamento de memória (operações in-place)
    for i in range(n):
        # num = (X[i] - X) ** 2
        num = X[i] - X
        np.square(num, out=num)  # Modifica o array temporário diretamente na memória
        
        # den = (X[i] + X) + 1e-10
        den = X[i] + X
        den += 1e-10             # Operação in-place
        
        num /= den               # Reutiliza o espaço de 'num' para a divisão
        D[i] = 0.5 * num.sum(axis=1)
        
    return D

def block_distance(name, X):
    m = METRIC.get(name, "euclidean")
    
    if m == "chi2":
        D = chi2_distance(X)
    else:
        # n_jobs=-1 paralisa o cálculo usando todas as CPUs disponíveis
        # astype(float32, copy=False) impede o desperdício de memória do padrão float64
        D = pairwise_distances(X, metric=m, n_jobs=-1).astype(np.float32, copy=False)
    
    # Normalização estritamente IN-PLACE (símbolos -= e /=)
    # Isso evita a criação de uma nova matriz temporária N x N na memória
    dmin, dmax = D.min(), D.max()
    D -= dmin
    D /= (dmax - dmin + 1e-10)   # Aplica diretamente sobre D e retorna
    
    return D

# 2. Execução do pipeline
# Nota: Esta linha ainda armazenará todas as matrizes simultaneamente.
dist = {k: block_distance(k, v) for k, v in norm_blocks.items()}
print("Matrizes de distância prontas (Otimizadas em float32):", list(dist.keys()))

Fusão tardia e recuperação

In [ ]:
DEFAULT_WEIGHTS = {"deep": 0.5, "color": 0.2, "gabor": 0.1,
                   "glcm": 0.1, "lbp": 0.05, "shape": 0.05}

def fuse(weights):
    keys = [k for k in weights if k in dist]
    w = np.array([weights[k] for k in keys])
    w = w / w.sum()
    D = sum(wi * dist[k] for wi, k in zip(w, keys))
    return D

def retrieve(query_idx, D, k=10):
    order = np.argsort(D[query_idx])
    order = order[order != query_idx]   # remove a própria consulta
    return order[:k]

Visualizar uma consulta

In [ ]:
CLASS_NAMES = class_cols  # nomes das colunas do CSV

def show_query(query_idx, D, k=8):
    res = retrieve(query_idx, D, k)
    fig, axes = plt.subplots(1, k + 1, figsize=(2.2 * (k + 1), 2.6))
    def load(i):
        return np.array(Image.open(os.path.join(IMAGES_DIR, ids[i] + ".jpg")))
    axes[0].imshow(load(query_idx)); axes[0].set_title(
        f"CONSULTA\n{CLASS_NAMES[labels[query_idx]]}", color="navy")
    axes[0].axis("off")
    for ax, ri in zip(axes[1:], res):
        ax.imshow(load(ri))
        ok = labels[ri] == labels[query_idx]
        ax.set_title(CLASS_NAMES[labels[ri]], color="green" if ok else "red")
        ax.axis("off")
    plt.tight_layout(); plt.show()

D_fused = fuse(DEFAULT_WEIGHTS)
q = np.random.choice(np.where(mask_valid)[0])
show_query(q, D_fused)

Avaliação

In [ ]:
def average_precision(rel):
    # rel: array booleano de relevância na ordem recuperada
    if rel.sum() == 0:
        return 0.0
    hits = np.cumsum(rel)
    prec_at_i = hits / (np.arange(len(rel)) + 1)
    return (prec_at_i * rel).sum() / rel.sum()

def evaluate(D, k=10, query_idxs=None):
    if query_idxs is None:
        query_idxs = np.where(mask_valid)[0]
    p_at_k, r_at_k, aps = [], [], []
    for qi in query_idxs:
        order = retrieve(qi, D, k=D.shape[0] - 1)   # ranking completo p/ mAP
        rel = (labels[order] == labels[qi])
        total_rel = (labels[mask_valid] == labels[qi]).sum() - 1
        p_at_k.append(rel[:k].mean())
        r_at_k.append(rel[:k].sum() / max(total_rel, 1))
        aps.append(average_precision(rel))
    return {"P@%d" % k: np.mean(p_at_k),
            "R@%d" % k: np.mean(r_at_k),
            "mAP": np.mean(aps)}

# amostra de consultas para ir rápido; use todas no resultado final
sample = np.random.choice(np.where(mask_valid)[0],
                          size=min(300, mask_valid.sum()), replace=False)
print(evaluate(D_fused, k=10, query_idxs=sample))

TFIDF

In [ ]:
import pickle
from sklearn.cluster import MiniBatchKMeans
from sklearn.preprocessing import normalize

DESC_CACHE = "sift_descriptors.pkl"
MAX_FEATURES = 300   # nº máximo de keypoints por imagem

def extract_local_descriptors(ids, images_dir, max_features=MAX_FEATURES):
    sift = cv2.SIFT_create(nfeatures=max_features)
    desc_list = []
    for n, i in enumerate(ids, 1):
        p = os.path.join(images_dir, i + ".jpg")
        img = cv2.imread(p, cv2.IMREAD_GRAYSCALE)
        if img is None:
            desc_list.append(np.empty((0, 128), np.float32)); continue
        img = cv2.resize(img, (256, 256))
        _, des = sift.detectAndCompute(img, None)
        desc_list.append(des if des is not None else np.empty((0, 128), np.float32))
        if n % 200 == 0:
            print(f"  SIFT {n}/{len(ids)}")
    return desc_list

if os.path.exists(DESC_CACHE):
    with open(DESC_CACHE, "rb") as f:
        desc_list = pickle.load(f)
    print("Descritores carregados do cache.")
else:
    desc_list = extract_local_descriptors(ids, IMAGES_DIR)   # demorado!
    with open(DESC_CACHE, "wb") as f:
        pickle.dump(desc_list, f)
print("Imagens com descritores:", sum(len(d) > 0 for d in desc_list))

Bag of visual words

In [ ]:
K_WORDS = 500   # tamanho do vocabulário visual

# amostra de descritores para treinar o k-means (treinar em todos é caro)
rng = np.random.default_rng(42)
pool = np.vstack([d for d in desc_list if len(d) > 0])
sample_idx = rng.choice(len(pool), size=min(100_000, len(pool)), replace=False)

kmeans = MiniBatchKMeans(n_clusters=K_WORDS, random_state=42,
                         batch_size=1000, n_init=3)
kmeans.fit(pool[sample_idx])
print("Vocabulário treinado:", K_WORDS, "palavras visuais")

def bovw_histogram(des):
    if len(des) == 0:
        return np.zeros(K_WORDS, dtype=np.float32)
    words = kmeans.predict(des)
    return np.bincount(words, minlength=K_WORDS).astype(np.float32)

tf = np.vstack([bovw_histogram(d) for d in desc_list])   # N x K_WORDS (frequência)
print("Matriz termo-documento (tf):", tf.shape)

Modelo vetorial

In [ ]:
N = tf.shape[0]
df = (tf > 0).sum(axis=0)                       # nº de imagens que contêm cada palavra
idf = np.log((N + 1) / (df + 1)) + 1.0          # IDF suavizado
tfidf = tf * idf
tfidf_n = normalize(tfidf, norm="l2").astype(np.float32)

D_tfidf = (1.0 - tfidf_n @ tfidf_n.T).astype(np.float32)   # distância cosseno
print("D_tfidf:", D_tfidf.shape)

bm25


In [ ]:
def bm25_distance(tf, k1=1.5, b=0.75):
    N, K = tf.shape
    df = (tf > 0).sum(axis=0)
    idf = np.log((N - df + 0.5) / (df + 0.5) + 1.0)
    doc_len = tf.sum(axis=1)
    avgdl = doc_len.mean() + 1e-9
    denom = tf + k1 * (1 - b + b * doc_len[:, None] / avgdl)
    W = (idf[None, :] * (tf * (k1 + 1)) / (denom + 1e-9)).astype(np.float32)  # pesos doc
    Q = (tf > 0).astype(np.float32)                                           # consulta
    S = Q @ W.T                                  # N x N : score BM25
    return (S.max() - S).astype(np.float32)      # score -> distância

D_bm25 = bm25_distance(tf)
print("D_bm25:", D_bm25.shape)

Avaliacao Geral

In [ ]:
def evaluate_multi(D, ks=(10, 20, 50, 100), query_idxs=None):
    if query_idxs is None:
        query_idxs = np.where(mask_valid)[0]
    pk = {f"P@{k}": [] for k in ks}
    aps = []
    for qi in query_idxs:
        order = retrieve(qi, D, k=D.shape[0] - 1)
        rel = (labels[order] == labels[qi])
        for k in ks:
            pk[f"P@{k}"].append(rel[:k].mean())
        aps.append(average_precision(rel))
    res = {m: float(np.mean(v)) for m, v in pk.items()}
    res["MAP"] = float(np.mean(aps))
    return res

def pr_curve(D, query_idxs=None, levels=np.linspace(0, 1, 11)):
    """Curva precision-recall interpolada (11 pontos), média sobre as consultas."""
    if query_idxs is None:
        query_idxs = np.where(mask_valid)[0]
    curves = []
    for qi in query_idxs:
        order = retrieve(qi, D, k=D.shape[0] - 1)
        rel = (labels[order] == labels[qi]).astype(int)
        total_rel = rel.sum()
        if total_rel == 0:
            continue
        hits = np.cumsum(rel)
        precision = hits / (np.arange(len(rel)) + 1)
        recall = hits / total_rel
        interp = [precision[recall >= r].max() if (recall >= r).any() else 0.0
                  for r in levels]
        curves.append(interp)
    return levels, np.mean(curves, axis=0)

Tabela Comparativa

In [ ]:
# usa a mesma amostra de consultas das seções anteriores
methods = {
    "TF-IDF (BoVW)": D_tfidf,
    "BM25 (BoVW)": D_bm25,
    "Fusão CBIR": D_fused,
}
rows = [{"método": name, **evaluate_multi(D, query_idxs=sample)}
        for name, D in methods.items()]
comparison = pd.DataFrame(rows)[
    ["método", "P@10", "P@20", "P@50", "P@100", "MAP"]
].sort_values("MAP", ascending=False)
display(comparison.reset_index(drop=True))

In [ ]:
plt.figure(figsize=(7, 5))
for name, D in methods.items():
    r, p = pr_curve(D, query_idxs=sample)
    plt.plot(r, p, marker="o", label=name)
plt.xlabel("Recall"); plt.ylabel("Precision")
plt.title("Curva Precision-Recall (interpolada)")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

Visualizar dispersão


In [ ]:
from sklearn.manifold import TSNE

def plot_tsne(X, title, perplexity=30, sample_size=1500, random_state=42):
    """Projeta X (N x d) em 2D com t-SNE e plota colorindo por classe."""
    rng = np.random.default_rng(random_state)
    valid = np.where(mask_valid)[0]
    if len(valid) > sample_size:
        valid = rng.choice(valid, size=sample_size, replace=False)

    Xs = np.asarray(X)[valid]
    ys = labels[valid]
    perp = min(perplexity, len(valid) - 1)

    X_2d = TSNE(n_components=2, random_state=random_state,
                perplexity=perp, init="pca").fit_transform(Xs)

    plt.figure(figsize=(12, 8))
    for c in np.unique(ys):
        m = ys == c
        plt.scatter(X_2d[m, 0], X_2d[m, 1],
                    label=CLASS_NAMES[c], alpha=0.6, s=18)
    plt.legend(); plt.title(title)
    plt.xlabel("t-SNE 1"); plt.ylabel("t-SNE 2")
    plt.tight_layout(); plt.show()
    return X_2d

_ = plot_tsne(tfidf_n, "ISIC 2018 — TF-IDF (Bag of Visual Words)")
_ = plot_tsne(norm_blocks["deep"], "ISIC 2018 — Deep features (ResNet50)")

Recuperação por classe


In [ ]:
# nome <-> índice
print("Classes:", {i: n for i, n in enumerate(CLASS_NAMES)})

CLASSE = "MEL"                       # troque pelo nome desejado
c = CLASS_NAMES.index(CLASSE)
idx_classe = np.where((labels == c) & mask_valid)[0]
print(f"{CLASSE}: {len(idx_classe)} imagens")

# uma consulta aleatória dessa classe (verde = recuperou da mesma classe)
show_query(np.random.choice(idx_classe), D_fused)

Metricas por classe

In [ ]:
def evaluate_per_class(D, max_queries=200):
    rng = np.random.default_rng(42)
    rows = []
    for c in np.unique(labels[mask_valid]):
        qidx = np.where((labels == c) & mask_valid)[0]
        if len(qidx) > max_queries:           # amostra para acelerar
            qidx = rng.choice(qidx, max_queries, replace=False)
        m = evaluate_multi(D, query_idxs=qidx)
        rows.append({"classe": CLASS_NAMES[c], "n_consultas": len(qidx), **m})
    return pd.DataFrame(rows).sort_values("MAP", ascending=False).reset_index(drop=True)

per_class = evaluate_per_class(D_fused)   # troque por D_tfidf / D_bm25 para comparar modelos
display(per_class)

Comparação
Final


In [ ]:
ax = per_class.set_index("classe")[["P@10", "P@50", "MAP"]].plot(
    kind="bar", figsize=(10, 5))
ax.set_ylabel("valor"); ax.set_title("Recuperação por classe — Fusão CBIR")
plt.xticks(rotation=0); plt.tight_layout(); plt.show()